# Star Schema Modeling (Databricks)

In [ ]:
from pyspark.sql.functions import coalesce, col, concat_ws, dayofmonth, md5, month, to_date, weekofyear, year

BUCKET_NAME = "ecommerce-data-pipeline-jayant"
CURATED_PATH = f"s3://{BUCKET_NAME}/curated/sales/"
DIM_PATH = f"s3://{BUCKET_NAME}/warehouse/dimensions/"
FACT_PATH = f"s3://{BUCKET_NAME}/warehouse/facts/"

df_sales = spark.read.parquet(CURATED_PATH)
df_sales.printSchema()

### Ensure Date and Numeric Types

In [ ]:
df_sales = (
    df_sales
    .withColumn("Order_Date", coalesce(col("Order_Date"), to_date(col("Order Date"), "MM/dd/yyyy")))
    .withColumn("Ship_Date", coalesce(col("Ship_Date"), to_date(col("Ship Date"), "MM/dd/yyyy")))
    .withColumn("Quantity", col("Quantity").cast("int"))
    .withColumn("Discount", col("Discount").cast("double"))
    .withColumn("region_id", md5(concat_ws("|", col("Region"), col("State"), col("City"))))
)

### Build Dimension Tables

In [ ]:
dim_customer = df_sales.select(
    col("Customer ID").alias("customer_id"),
    col("Customer Name").alias("customer_name"),
    col("Segment").alias("segment"),
    col("Country").alias("country")
).dropDuplicates(["customer_id"])

dim_product = df_sales.select(
    col("Product ID").alias("product_id"),
    col("Product Name").alias("product_name"),
    col("Category").alias("category"),
    col("Sub-Category").alias("sub_category")
).dropDuplicates(["product_id"])

dim_region = df_sales.select(
    col("region_id"),
    col("Region").alias("region"),
    col("Country").alias("country"),
    col("State").alias("state"),
    col("City").alias("city"),
    col("Postal Code").alias("postal_code")
).dropDuplicates(["region_id"])

dim_date = (
    df_sales
    .select(col("Order_Date").alias("order_date"))
    .filter(col("order_date").isNotNull())
    .dropDuplicates()
    .withColumn("year", year("order_date"))
    .withColumn("month", month("order_date"))
    .withColumn("day", dayofmonth("order_date"))
    .withColumn("week", weekofyear("order_date"))
)

### Build Fact Table and Write Warehouse Parquet

In [ ]:
fact_sales = df_sales.select(
    col("Order ID").alias("order_id"),
    col("Customer ID").alias("customer_id"),
    col("Product ID").alias("product_id"),
    col("region_id"),
    col("Order_Date").alias("order_date"),
    col("Ship_Date").alias("ship_date"),
    col("Ship Mode").alias("ship_mode"),
    col("Sales").alias("sales"),
    col("Quantity").alias("quantity"),
    col("Discount").alias("discount"),
    col("Profit").alias("profit")
)

dim_customer.write.mode("overwrite").parquet(DIM_PATH + "dim_customer/")
dim_product.write.mode("overwrite").parquet(DIM_PATH + "dim_product/")
dim_region.write.mode("overwrite").parquet(DIM_PATH + "dim_region/")
dim_date.write.mode("overwrite").parquet(DIM_PATH + "dim_date/")
fact_sales.write.mode("overwrite").parquet(FACT_PATH + "fact_sales/")